In [1]:
import sys, pathlib, warnings
warnings.filterwarnings('ignore')
sys.path.insert(0, r'/Users/alfredtang/Documents/Projects/gen-e2/gen-e2-data-analysis/artifacts/ps-004-headcount-forecasting/src')
sys.path.insert(0, r'/Users/alfredtang/Documents/Projects/gen-e2/gen-e2-data-analysis/artifacts/ps-004-headcount-forecasting/src/models')
import polars as pl, joblib
from feature_engineering import build_features
from linear_model import train_linear_models
from arima_model import train_arima_models
from model_evaluation import evaluate_models, select_champions, generate_forecast, build_model_registry
print("All imports OK")

All imports OK


In [2]:
features = pl.read_parquet(r'/Users/alfredtang/Documents/Projects/gen-e2/gen-e2-data-analysis/artifacts/ps-004-headcount-forecasting/data/3_interim/features.parquet')
print("Features shape:", features.shape)
print(features.head(8))

Features shape: (48, 6)
shape: (8, 6)
┌────────────┬──────┬───────┬────────────┬───────┬───────┐
│ profession ┆ year ┆ count ┆ year_index ┆ lag_1 ┆ lag_2 │
│ ---        ┆ ---  ┆ ---   ┆ ---        ┆ ---   ┆ ---   │
│ cat        ┆ i32  ┆ i32   ┆ i32        ┆ i32   ┆ i32   │
╞════════════╪══════╪═══════╪════════════╪═══════╪═══════╡
│ doctors    ┆ 2006 ┆ 6931  ┆ 0          ┆ null  ┆ null  │
│ doctors    ┆ 2007 ┆ 7464  ┆ 1          ┆ 6931  ┆ null  │
│ doctors    ┆ 2008 ┆ 7841  ┆ 2          ┆ 7464  ┆ 6931  │
│ doctors    ┆ 2009 ┆ 8323  ┆ 3          ┆ 7841  ┆ 7464  │
│ doctors    ┆ 2010 ┆ 9030  ┆ 4          ┆ 8323  ┆ 7841  │
│ doctors    ┆ 2011 ┆ 9646  ┆ 5          ┆ 9030  ┆ 8323  │
│ doctors    ┆ 2012 ┆ 10225 ┆ 6          ┆ 9646  ┆ 9030  │
│ doctors    ┆ 2013 ┆ 10953 ┆ 7          ┆ 10225 ┆ 9646  │
└────────────┴──────┴───────┴────────────┴───────┴───────┘


In [3]:
models_dir = pathlib.Path(r'/Users/alfredtang/Documents/Projects/gen-e2/gen-e2-data-analysis/artifacts/ps-004-headcount-forecasting/models')
linear_models = {}
for pkl in sorted(models_dir.glob('*_linear_*.pkl')):
    prof = pkl.stem.rsplit('_linear_', 1)[0]
    linear_models[prof] = joblib.load(pkl)
    print(f"Loaded linear: {prof}  coef={linear_models[prof].coef_[0]:.2f}")
print("Linear models:", list(linear_models.keys()))

Loaded linear: doctors  coef=620.83
Loaded linear: nurses  coef=2080.20
Loaded linear: pharmacists  coef=156.46
Loaded linear: physiotherapists  coef=149.50
Linear models: ['doctors', 'nurses', 'pharmacists', 'physiotherapists']


In [4]:
arima_models = train_arima_models(features, holdout_years=3)
print("ARIMA models trained:")
for prof, (model, order) in arima_models.items():
    print(f"  {prof}: order={order}")

WARNING [doctors] ARIMA(1, 1, 2) convergence warning: Maximum Likelihood optimization failed to converge. Check mle_retvals
WARNING [doctors] ARIMA(2, 0, 2) convergence warning: Maximum Likelihood optimization failed to converge. Check mle_retvals
WARNING [doctors] ARIMA(2, 1, 2) convergence warning: Maximum Likelihood optimization failed to converge. Check mle_retvals
INFO [doctors] selected ARIMA(1, 1, 2) with AIC=131.077


WARNING [nurses] ARIMA(2, 1, 2) convergence warning: Maximum Likelihood optimization failed to converge. Check mle_retvals
INFO [nurses] selected ARIMA(1, 1, 2) with AIC=160.045
WARNING [pharmacists] ARIMA(2, 0, 1) convergence warning: Maximum Likelihood optimization failed to converge. Check mle_retvals
WARNING [pharmacists] ARIMA(2, 0, 2) convergence warning: Maximum Likelihood optimization failed to converge. Check mle_retvals
WARNING [pharmacists] ARIMA(2, 1, 1) convergence warning: Maximum Likelihood optimization failed to converge. Check mle_retvals
WARNING [pharmacists] ARIMA(2, 1, 2) convergence warning: Maximum Likelihood optimization failed to converge. Check mle_retvals
INFO [pharmacists] selected ARIMA(1, 1, 0) with AIC=108.662


WARNING [physiotherapists] ARIMA(1, 1, 1) convergence warning: Maximum Likelihood optimization failed to converge. Check mle_retvals
WARNING [physiotherapists] ARIMA(1, 1, 2) convergence warning: Maximum Likelihood optimization failed to converge. Check mle_retvals
WARNING [physiotherapists] ARIMA(2, 0, 0) convergence warning: Maximum Likelihood optimization failed to converge. Check mle_retvals
WARNING [physiotherapists] ARIMA(2, 0, 1) convergence warning: Maximum Likelihood optimization failed to converge. Check mle_retvals
WARNING [physiotherapists] ARIMA(2, 0, 2) convergence warning: Maximum Likelihood optimization failed to converge. Check mle_retvals
WARNING [physiotherapists] ARIMA(2, 1, 0) convergence warning: Maximum Likelihood optimization failed to converge. Check mle_retvals
WARNING [physiotherapists] ARIMA(2, 1, 1) convergence warning: Maximum Likelihood optimization failed to converge. Check mle_retvals
WARNING [physiotherapists] ARIMA(2, 1, 2) convergence warning: Maximu

In [5]:
eval_df = evaluate_models(features, linear_models, arima_models, holdout_years=3)
print("Model comparison:")
print(eval_df.sort(['profession', 'model_type']))

Model comparison:
shape: (8, 7)
┌──────────────────┬────────────┬────────┬───────────┬───────────┬─────────────┬───────────────────┐
│ profession       ┆ model_type ┆ mape   ┆ mae       ┆ rmse      ┆ arima_order ┆ arima_is_fallback │
│ ---              ┆ ---        ┆ ---    ┆ ---       ┆ ---       ┆ ---         ┆ ---               │
│ str              ┆ str        ┆ f64    ┆ f64       ┆ f64       ┆ str         ┆ bool              │
╞══════════════════╪════════════╪════════╪═══════════╪═══════════╪═════════════╪═══════════════════╡
│ doctors          ┆ arima      ┆ 1.5406 ┆ 215.4938  ┆ 244.6269  ┆ (1, 1, 2)   ┆ false             │
│ doctors          ┆ linear     ┆ 2.2538 ┆ 314.7303  ┆ 346.8399  ┆ null        ┆ null              │
│ nurses           ┆ arima      ┆ 2.9819 ┆ 1262.6283 ┆ 1373.8763 ┆ (1, 1, 2)   ┆ false             │
│ nurses           ┆ linear     ┆ 8.6084 ┆ 3639.6727 ┆ 3817.8366 ┆ null        ┆ null              │
│ pharmacists      ┆ arima      ┆ 4.1095 ┆ 135.1503  ┆ 149.

In [6]:
registry_df = select_champions(eval_df)
print("Champion models per profession:")
print(registry_df.select(['profession', 'model_type', 'mape', 'mape_target_met']))

Champion models per profession:
shape: (4, 4)
┌──────────────────┬────────────┬────────┬─────────────────┐
│ profession       ┆ model_type ┆ mape   ┆ mape_target_met │
│ ---              ┆ ---        ┆ ---    ┆ ---             │
│ str              ┆ str        ┆ f64    ┆ bool            │
╞══════════════════╪════════════╪════════╪═════════════════╡
│ doctors          ┆ arima      ┆ 1.5406 ┆ true            │
│ nurses           ┆ arima      ┆ 2.9819 ┆ true            │
│ pharmacists      ┆ linear     ┆ 2.0482 ┆ true            │
│ physiotherapists ┆ arima      ┆ 2.326  ┆ true            │
└──────────────────┴────────────┴────────┴─────────────────┘


In [7]:
# Build champion_models dict in the format expected by generate_forecast
# generate_forecast expects: {profession: {"model": model_obj, "model_type": str}}
champion_models = {}
for row in registry_df.iter_rows(named=True):
    prof = row['profession']
    mt = row['model_type']
    if mt == 'arima' and prof in arima_models:
        champion_models[prof] = {"model": arima_models[prof][0], "model_type": "arima"}
    else:
        champion_models[prof] = {"model": linear_models[prof], "model_type": "linear"}

forecast_df = generate_forecast(features, champion_models, horizon=5)
print("5-year forecast (2020-2024):")
print(forecast_df.sort(['profession', 'year']))

# Update contract files with champion selection
full_registry = build_model_registry(registry_df, forecast_df)
import pathlib
METRICS = pathlib.Path(r'/Users/alfredtang/Documents/Projects/gen-e2/gen-e2-data-analysis/artifacts/ps-004-headcount-forecasting/results/metrics')
EXPORTS = pathlib.Path(r'/Users/alfredtang/Documents/Projects/gen-e2/gen-e2-data-analysis/artifacts/ps-004-headcount-forecasting/results/exports')

eval_df.select(['profession','model_type','mape','mae','rmse']).write_csv(METRICS / 'model_comparison.csv')
full_registry.write_csv(METRICS / 'model_registry.csv')
forecast_df.write_csv(EXPORTS / 'forecast_table.csv')

for p in [METRICS / 'model_comparison.csv', METRICS / 'model_registry.csv', EXPORTS / 'forecast_table.csv']:
    print(f"  {p.name}: {p.stat().st_size}B")

5-year forecast (2020-2024):
shape: (20, 6)
┌──────────────────┬──────┬────────────────┬────────────┬────────────┬────────────┐
│ profession       ┆ year ┆ forecast_count ┆ model_type ┆ lower_95   ┆ upper_95   │
│ ---              ┆ ---  ┆ ---            ┆ ---        ┆ ---        ┆ ---        │
│ str              ┆ i64  ┆ f64            ┆ str        ┆ f64        ┆ f64        │
╞══════════════════╪══════╪════════════════╪════════════╪════════════╪════════════╡
│ doctors          ┆ 2020 ┆ 13442.823      ┆ arima      ┆ 13253.3873 ┆ 13632.2587 │
│ doctors          ┆ 2021 ┆ 14025.8322     ┆ arima      ┆ 13603.7277 ┆ 14447.9368 │
│ doctors          ┆ 2022 ┆ 14608.8263     ┆ arima      ┆ 14024.1442 ┆ 15193.5083 │
│ doctors          ┆ 2023 ┆ 15191.8052     ┆ arima      ┆ 14465.8855 ┆ 15917.7248 │
│ doctors          ┆ 2024 ┆ 15774.769      ┆ arima      ┆ 14918.3101 ┆ 16631.2279 │
│ …                ┆ …    ┆ …              ┆ …          ┆ …          ┆ …          │
│ physiotherapists ┆ 2020 ┆ 1836